In [3]:
import math
import os
import torch
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import warnings
import pandas_ta as ta

from torch.optim              import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from transformers             import EarlyStoppingCallback, Trainer, TrainingArguments, set_seed
from tsfm_public              import TimeSeriesForecastingPipeline 
from tsfm_public              import TimeSeriesPreprocessor
from tsfm_public              import TinyTimeMixerForPrediction
from tsfm_public              import TrackingCallback
from tsfm_public              import count_parameters
from tsfm_public              import get_datasets


from tsfm_public.toolkit.time_series_preprocessor import prepare_data_splits
from sklearn.metrics import precision_score, recall_score, f1_score
import ast
warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)

from sklearn.metrics import precision_score, recall_score, f1_score

def metrics(actual, prediction):
    a = np.stack(actual).flatten()
    p = np.stack(prediction).flatten()
    mask = ~np.isnan(a) & ~np.isnan(p)
    a, p = a[mask], p[mask]

    # errors
    mae  = np.mean(np.abs(a - p))
    rmse = np.sqrt(np.mean((a - p)**2))
    mape = np.mean(np.abs((a - p) / (a + 1e-8))) * 100

    # direction
    actual_diff = np.sign(np.diff(a))
    pred_diff   = np.sign(np.diff(p))
    hit_rate = np.mean(actual_diff == pred_diff)

    actual_up = actual_diff > 0
    pred_up   = pred_diff   > 0
    precision = precision_score(actual_up, pred_up, zero_division=0)
    recall    = recall_score(actual_up, pred_up, zero_division=0)
    f1        = f1_score(actual_up, pred_up, zero_division=0)

    return dict(mae=mae, rmse=rmse, mape=mape,
                hit_rate=hit_rate,
                precision=precision, recall=recall, f1=f1)



device             = "cuda" if torch.cuda.is_available() else "cpu"
load_path          =  "ibm-granite/granite-timeseries-ttm-r2"
os.makedirs('final_data', exist_ok=True)


tickers = ["AAPL",
           "TSLA",
           "XOM",
           "SPY",
           "JNJ",
           "AMD",
           "PG"
          ]

event_titles = ["AAPL – Crash & Rebound (2020-03-10)",
                "TSLA – High-Beta Cooling (2021-01-15)",
                "XOM – Oil Cycle Peak (2022-06-01)",
                "SPY – Drawdown Chop (2022-09-15)",
                "JNJ – Low-Volatility Stretch (2019-08-01)",
                "AMD – Tech Selloff (2018-10-10)",
                "PG – Macro-Irrelevant Calm (2015-06-15)"
                ]

starts = ["2019-09-02",
          "2020-07-01",
          "2021-10-01",
          "2022-01-03",
          "2018-11-01",
          "2018-04-02",
          "2014-12-01"
         ]

ends   = ["2020-06-01",
          "2021-06-30",
          "2022-09-30",
          "2022-12-30",
          "2020-01-01",
          "2019-03-29",
          "2016-01-01"
         ]



def create_test_sets():
    test_sets = []
    for ticker in tickers:
        df = pd.read_csv(f"./final_data/{ticker}.csv", parse_dates=["date"])
        _, _, test_df = prepare_data_splits(df,context_length=512,split_config={"train": 0.6,"test": 0.4})
        test_sets.append(test_df)
    return test_sets

def predict_series(df, test_sets, pipeline, _type, start, end, event_title):
    for test_df,ticker in zip(test_sets,tickers):
        forecast = pipeline(test_df)
        forecast["date"]     = pd.to_datetime(forecast["date"])
        forecast["y_true"]   = forecast["close"].str[0]
        forecast["y_pred"]   = forecast["close_prediction"].str[0]
        forecast["residual"] = forecast["y_true"] - forecast["y_pred"]
        output_path          = os.path.join('fewshot_results', f'{_type}_predict_{ticker}.csv')
        forecast.to_csv(output_path, index=False)
        
        forecast         = pipeline(df)
        start            = pd.to_datetime(start)
        end              = pd.to_datetime(end)
        mask             = (forecast["date"] >= start) & (forecast["date"] <= end)
        event            = forecast.loc[mask]
        event["y_true"]  = event["close"].str[0]
        event["y_pred"]  = event["close_prediction"].str[0]
        output_path      = os.path.join('fewshot_results',  f'{_type}_predict_{ticker}_event.csv')
        event.to_csv(output_path, index=False)


idx = 0


for ticker in tickers:
    df = pd.read_csv(f"./processed_data/{ticker}.csv", parse_dates=["date"])
    df.set_index("date", inplace=True)

    df["sma_20"] = ta.sma(df["close"], length=20)
    df["ema_20"] = ta.ema(df["close"], length=20)
    df["wma_20"] = ta.wma(df["close"], length=20)
    df["tema_20"] = ta.tema(df["close"], length=20)
    bbands = ta.bbands(df["close"], length=20)
    df["bb_upper"] = bbands["BBU_20_2.0"]
    df["bb_lower"] = bbands["BBL_20_2.0"]
    df.dropna(inplace=True)
    df.reset_index(inplace=True)
    df.to_csv(os.path.join( "./final_data", f"{ticker}.csv"), index=False)

test_sets = create_test_sets()


column_specifiers = {
    "timestamp_column": "date",
    "id_columns": [],
    "target_columns": ["close"],
    "control_columns": [
        "open", "high", "low",
        "sma_20", "ema_20", "wma_20", "tema_20",
        "bb_upper", "bb_lower"
    ]
}


preprocessor = TimeSeriesPreprocessor(
    **column_specifiers,
    context_length     = 512,
    prediction_length  = 96,
    scaling            = True,
    encode_categorical = False,
    scaler_type        = "standard",
)

model = TinyTimeMixerForPrediction.from_pretrained(
    load_path , 
    num_input_channels             = preprocessor.num_input_channels,
    prediction_channel_indices     = preprocessor.prediction_channel_indices,
    exogenous_channel_indices      = preprocessor.exogenous_channel_indices,
    fcm_use_mixer                  = False,
    enable_forecast_channel_mixing = False,
    decoder_mode                   = "direct",
)

test_sets = create_test_sets()

os.makedirs('finalmodel', exist_ok=True)


logs      = []

tickers2 = ["AAPL",
           "TSLA",
           "XOM",
           "SPY",
           "JNJ",
           "AMD",
           "PG"
          ]

for ticker,start,end,event_title in zip(tickers2,starts,ends,event_titles):
    
    df = pd.read_csv(f"./final_data/{ticker}.csv", parse_dates=["date"])
    
    preprocessor.train(df)

    train_df, valid_df, test_df = prepare_data_splits(
        df,
        context_length=512,
        split_config={"train": 0.6,"test": 0.4}
    )
    
    for param in model.backbone.parameters():
     param.requires_grad = False

    train_set, valid_set, test_set = get_datasets(
        preprocessor,
        df,
        {"train": 0.6,"test": 0.4},
        fewshot_fraction    = 0.7,
        fewshot_location    = "first",
        use_frequency_token = model.config.resolution_prefix_tuning,
    )

    learning_rate  = 0.00225
    num_epochs     = 5
    patience       = 10
    batch_size     = 64

    args = TrainingArguments(
        output_dir                  = os.path.join('fewshot_full_results', "output"),
        overwrite_output_dir        = True,
        learning_rate               = learning_rate,
        num_train_epochs            = num_epochs,
        do_eval                     = True,
        eval_strategy               = "epoch",
        per_device_train_batch_size = batch_size,
        per_device_eval_batch_size  = batch_size,
        dataloader_num_workers      = 4,
        report_to                   = None,
        save_strategy               = "epoch",
        logging_strategy            = "epoch",
        save_total_limit            = 1,
        logging_dir                 = os.path.join('fewshot_full_results', "logs"),  
        load_best_model_at_end      = True,  
        metric_for_best_model       = "eval_loss",  
        greater_is_better           = False,  
        use_cpu                     = device != "cuda",
    )

    early_stopping_callback = EarlyStoppingCallback(
        early_stopping_patience=patience,
        early_stopping_threshold=0.00001, 
        )
    
    tracking_callback = TrackingCallback()

    optimizer = AdamW(model.parameters(), lr=learning_rate)
    scheduler = OneCycleLR(optimizer, learning_rate, epochs=num_epochs, steps_per_epoch=math.ceil(len(train_set) / (batch_size)),)

    trainer = Trainer(
        model         = model,
        args          = args,
        train_dataset = train_set,
        eval_dataset  = valid_set,
        callbacks     = [early_stopping_callback, tracking_callback],
        optimizers    = (optimizer, scheduler),
    )
    
    trainer.train()



pipeline = TimeSeriesForecastingPipeline(
    model,
    device            = device, 
    feature_extractor = preprocessor,
    batch_size        = batch_size,
)


i = 0
for ticker,start,end,event_title in zip(tickers,starts,ends,event_titles):
    df = pd.read_csv(f"./final_data/{ticker}.csv", parse_dates=["date"])
    test_df = test_sets[i]
    forecast = pipeline(test_df)
    forecast["date"]     = pd.to_datetime(forecast["date"])
    forecast["y_true"]   = forecast["close"].str[0]
    forecast["y_pred"]   = forecast["close_prediction"].str[0]
    forecast['y_pred'] = forecast['y_pred'].shift(-1)
    forecast["residual"] = forecast["y_true"] - forecast["y_pred"]
    output_path          = os.path.join('finalmodel', f'{ticker}_final_results.csv')
    forecast.to_csv(output_path, index=False)
    
    forecast         = pipeline(df)
    start            = pd.to_datetime(start)
    end              = pd.to_datetime(end)
    mask             = (forecast["date"] >= start) & (forecast["date"] <= end)
    event            = forecast.loc[mask]
    event["y_true"]  = event["close"].str[0]
    event["y_pred"]  = event["close_prediction"].str[0]
    event['y_pred'] = event['y_pred'].shift(-1)
    output_path      = os.path.join('finalmodel', f'{ticker}_final_event_results.csv')
    event.to_csv(output_path, index=False)
    i += 1

results = []

for ticker in tickers:
    df = pd.read_csv(f'finalmodel/{ticker}_final_results.csv')
    df = df.dropna(subset=["y_true", "y_pred"])
    m = metrics(df["y_true"].values, df["y_pred"].values)
    m["ticker"] = ticker
    results.append(m)


fewshot_df = pd.DataFrame(results).set_index("ticker")
output_path = os.path.join("finalmodel", "final_output.csv")
fewshot_df.to_csv(output_path)
print('\n\n',f'--------FINETUNE ON ALL--------')
print(fewshot_df)

results = []

for ticker in tickers:
    df = pd.read_csv(f'finalmodel/{ticker}_final_event_results.csv')
    df = df.dropna(subset=["y_true", "y_pred"])
    m = metrics(df["y_true"].values, df["y_pred"].values)
    m["ticker"] = ticker
    results.append(m)


fewshot_df = pd.DataFrame(results).set_index("ticker")
output_path = os.path.join("finalmodel", "final_event_output.csv")
fewshot_df.to_csv(output_path)
print(fewshot_df)

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [2]:
!pip install tf-keras

   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ------------------ --------------------- 0.8/1.7 MB 4.8 MB/s eta 0:00:01
   ---------------------------------------- 1.7/1.7 MB 4.5 MB/s  0:00:00
